In [1]:
import sys
sys.path.insert(0,'../scripts')
from pycap_for_PESTPP_MOU import postprocess_MOU, plot_pareto, prep_for_viz, create_viz_app, plot_pareto_with_scenarios
from pathlib import Path
from ipywidgets import interact, fixed
import yaml

# Read in one of the configuration files and postprocess it. 
### The `config_file` should be your `secnario_name` + `".yml"` as assigned in the `05_MOU_setup_run.ipynb` notebook. 

In [2]:
d_con = ['depletion_q_baseline_0.0_1.0_0.2',  'depletion_q_Tmin10per_0.0_1.0_0.2',  'depletion_q_Tplus10per_0.0_1.0_0.2']    #1
f_con = ['fish_dollars_baseline_0.0_1.0_0.2', 'fish_dollars_Tmin10per_0.0_1.0_0.2', 'fish_dollars_Tplus10per_0.0_1.0_0.2']   #0
configs = d_con + f_con

dof = 1
i = 0
scenario_name = d_con[i] if dof else f_con[i]
print(scenario_name)

config_file = scenario_name + '.yml'
with open(Path('./configurations') / config_file,'r') as ifp:
    inpars = yaml.safe_load(ifp)
inpars

depletion_q_baseline_0.0_1.0_0.2


{'depletion_potential_threshold': 0.2,
 'objectives': 'depletion_q',
 'pump_lbound_fraction': 0.0,
 'pump_ubound_fraction': 1.0,
 'ref_q': 8.6,
 'run_path': '../pycap_runs/pycap_pest/run_depletion_q_baseline_0.0_1.0_0.2',
 'scenario_name': 'depletion_q_baseline_0.0_1.0_0.2'}

### Below, the `config_file` is processed, beginning the postprocessing

In [3]:
run_path = Path(inpars['run_path'])
run_name = inpars['scenario_name']
pareto_df = postprocess_MOU(run_name, run_path)

In [4]:
pareto_df

,generation,member,Depletion (cfs),Total Pumping (cfs),nsga2_front,nsga2_crowding_distance,spea2_unconstrained_fitness,spea2_constrained_fitness,is_feasible,feasible_distance
0,0,10,4.89771,10.224827,1,1.000000e+30,0.008043,0.008043,1,-999
1,0,1,4.43955,8.759672,1,1.000000e+30,0.005179,0.005179,1,-999
2,0,12,4.82867,10.124990,1,3.410520e-01,0.012575,0.012575,1,-999
3,0,36,4.69714,9.952275,1,3.165240e-01,0.008043,0.008043,1,-999
4,0,30,4.63622,9.539204,1,2.119700e-01,0.016071,0.016071,1,-999
...,...,...,...,...,...,...,...,...,...,...
5627,50,gen=36_member=1334_pso,2.99219,6.237041,1,1.746380e-03,0.013341,0.013341,1,-999
5628,50,gen=23_member=858_pso,2.94751,6.131478,1,5.155960e-04,2.012280,2.012280,1,-999
5630,50,gen=50_member=1866_pso,3.28499,6.920903,1,1.611150e-02,0.007730,0.007730,1,-999
5639,50,gen=50_member=1871_pso,4.27059,8.820496,1,6.169430e-03,0.014589,0.014589,1,-999


### Next the evolution of the pareto frontier is plotted, over MOU iterations. Check out how the algorithm converges on a well-defined pareto tradeoff frontier

In [5]:
interact(plot_pareto,  currgen=(0,50,1), pareto_df=fixed(pareto_df));

interactive(children=(IntSlider(value=25, description='currgen', max=50), Output()), _dom_classes=('widget-int…

### Where do your scenarios line up? You can provide a list of scenarios that you ran in notebook #3 in the following function to plot a red "x" where each scenario plots up

### NOTE: This only works for the fish vs. receipts case!

In [6]:
dollars=False
# if 'dollar' in inpars['objectives']:
#     dollars = True
# if dollars:
#     plot_pareto_with_scenarios(pareto_df, scenarios=['LPR_Single_Run','LPR_Single_Run70','LPR_Single_Run90'])

### Finally, there is an interactive plot with the pareto frontier on the left and a map of corresponding "optimal" pumping rates on the right. 

### Note - if you get an error about ports being in conflict, change the `port=4242` argument to a different number than `4242`

In [7]:
pareto_df_final, dv_df =  prep_for_viz(pareto_df, 50, run_path, run_name, dollar_objective=dollars)
# app = create_viz_app(pareto_df_final, dv_df)
# app.run(port=4242)

### check out `pareto_df_final` and `dv_df`. What's in these datasets? What further analysis can you perform with this information?

In [8]:
dv_df.T.sample(10)

,well_1013,well_1302,well_1323,well_1486,well_1584,well_1589,well_1643,well_1683,well_1860,well_23610,...,well_93143,well_93349,well_93422,well_93423,well_93424,well_93469,well_93832,well_94302,well_94988,well_95068
gen=11_member=402_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
12,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=43_member=1609_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=23_member=858_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=7_member=260_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=30_member=1124_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=49_member=1852_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=38_member=1427_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
gen=17_member=644_pso,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [9]:
pareto_df_final.columns

Index(['generation', 'member', 'Depletion (cfs)', 'Total Pumping (cfs)',
       'nsga2_front', 'nsga2_crowding_distance', 'spea2_unconstrained_fitness',
       'spea2_constrained_fitness', 'is_feasible', 'feasible_distance'],
      dtype='object')

In [10]:
pareto_df_final.sample(10)

,generation,member,Depletion (cfs),Total Pumping (cfs),nsga2_front,nsga2_crowding_distance,spea2_unconstrained_fitness,spea2_constrained_fitness,is_feasible,feasible_distance
5502,50,gen=27_member=1020_pso,4.47218,9.215810,1,0.014295,2.011570,2.011570,1,-999
5536,50,gen=41_member=1550_pso,3.84942,8.050210,1,0.011170,0.019505,0.019505,1,-999
5533,50,gen=44_member=1663_pso,3.89374,8.140510,1,0.011292,0.013633,0.013633,1,-999
5463,50,gen=16_member=602_pso,2.67808,5.390557,1,0.019311,0.007517,0.007517,1,-999
5603,50,gen=12_member=452_pso,4.12930,8.563140,1,0.005612,0.016447,0.016447,1,-999
5597,50,gen=21_member=768_pso,2.33785,4.228388,1,0.006238,0.006436,0.006436,1,-999
5543,50,gen=31_member=1158_pso,3.60475,7.559871,1,0.010049,0.012606,0.012606,1,-999
5558,50,gen=31_member=1151_pso,3.57315,7.506377,1,0.009113,0.011711,0.011711,1,-999
5623,50,gen=42_member=1565_pso,4.34987,8.993901,1,0.002731,0.014112,0.014112,1,-999
5532,50,28,4.66154,9.644789,1,0.011311,1.005740,1.005740,1,-999


In [11]:
# If we want to, we can save out the pareto tradeoff objectives results to a file. 
# Just change the name below and it will be a new copy!
pareto_df_final[[pareto_df_final.columns[3], pareto_df_final.columns[2],'member']].to_csv(f'{scenario_name}_tradeoff.csv')